# MIMIC-CXR 多模态胸片 + 报告解读 Notebook

本 Notebook 将现有的 `multimodal_cxr_report.py` 脚本迁移为交互式环境，方便：
- 逐步调试多模态（图像 + 文本） vs 纯文本 LLM 的解读流程；
- 按需修改提示词、模型配置和数据路径；
- 在单独单元格中查看中间结果、绘图或导出统计结果。

In [7]:
# 安装最新版本
!pip install zai-sdk
# 或指定版本
!pip install zai-sdk==0.2.2

In [8]:
# 1. 创建并配置 Notebook 环境单元格

import os
import sys
from pathlib import Path

# 项目根目录：根据当前 Notebook 所在位置自动推断
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent  # srtp-medical-ai
DATA_ROOT = PROJECT_ROOT.parent / "SRTP_Data"

print("当前工作目录:", CURRENT_DIR)
print("项目根目录:", PROJECT_ROOT)
print("数据目录:", DATA_ROOT)

# 确保可以 import 项目内的模块
if str(PROJECT_ROOT / "code") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "code"))


当前工作目录: d:\Documents\SRTP_main\srtp-medical-ai\code
项目根目录: d:\Documents\SRTP_main\srtp-medical-ai
数据目录: d:\Documents\SRTP_main\SRTP_Data


In [9]:
# 2. 从现有 .py 文件导入代码（可选）
# 你可以直接使用 %run 运行原脚本，或者只当它是参考实现。

script_path = PROJECT_ROOT / "code" / "multimodal_cxr_report.py"
print("脚本路径:", script_path)

if script_path.exists():
    print("你可以运行下面一行来直接执行原脚本（带命令行参数）:")
    print("!python", script_path, "--help")
else:
    print("警告: 找不到原始脚本文件，请检查路径。")


脚本路径: d:\Documents\SRTP_main\srtp-medical-ai\code\multimodal_cxr_report.py
你可以运行下面一行来直接执行原脚本（带命令行参数）:
!python d:\Documents\SRTP_main\srtp-medical-ai\code\multimodal_cxr_report.py --help


In [10]:
# 3. 将脚本中的函数和类拆分到独立单元格


import json
import re
from dataclasses import dataclass
from typing import Dict, List, Optional


import pandas as pd
from PIL import Image
from dotenv import load_dotenv


# 14 类 CheXpert 标签名称（与 chexpert.csv 对应）
CHEXPERT_LABELS: List[str] = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "Pleural Other",
    "Support Devices",
    "No Finding",
]


@dataclass
class ModelConfig:
    """模型与路径配置（可在 Notebook 中直接修改实例字段）"""


    # GLM-4.6V 模型名称（使用官方文档推荐的 `glm-4.6v`）
    glm_model_multimodal: str = "glm-4.6v"
    glm_model_text: str = "glm-4.6v"


    # 数据路径配置
    data_root: str = str(DATA_ROOT)
    chexpert_csv: str = str(DATA_ROOT / "mimic-cxr-2.0.0-chexpert.csv")




def parse_ids_from_path(image_path: str) -> Optional[Dict[str, str]]:
    """从 MIMIC-CXR-JPG 路径中解析 subject_id / study_id。


    目录结构类似于: .../files/p10/p10000032/s56699142/<random-name>.jpg
    其中 p10000032 表示 subject_id=10000032，s56699142 表示 study_id=56699142。
    """


    path = Path(image_path)
    subject_id: Optional[str] = None
    study_id: Optional[str] = None


    for part in path.parts:
        if re.match(r"p\d+", part):
            subject_id = part[1:]
        if re.match(r"s\d+", part):
            study_id = part[1:]


    if not subject_id or not study_id:
        return None
    return {"subject_id": subject_id, "study_id": study_id}




def load_chexpert_labels(config: ModelConfig) -> pd.DataFrame:
    df = pd.read_csv(config.chexpert_csv)
    missing = [c for c in CHEXPERT_LABELS if c not in df.columns]
    if missing:
        raise ValueError(f"CheXpert CSV 缺少列: {missing}")
    return df




def get_labels_for_study(df: pd.DataFrame, subject_id: str, study_id: str) -> Dict[str, Optional[float]]:
    sub_id = int(subject_id)
    stu_id = int(study_id)
    row = df[(df["subject_id"] == sub_id) & (df["study_id"] == stu_id)]
    if row.empty:
        return {}
    row = row.iloc[0]
    labels: Dict[str, Optional[float]] = {}
    for lab in CHEXPERT_LABELS:
        val = row.get(lab)
        labels[lab] = float(val) if pd.notna(val) else None
    return labels



In [11]:
# 3-2. 模型调用与结果处理函数


import base64
from zai import ZhipuAiClient


def get_glm_client() -> ZhipuAiClient:
    """初始化 GLM-4.6V 客户端，从 .env 中读取 GLM_API_KEY。"""


    load_dotenv(override=False)
    api_key = os.getenv("GLM_API_KEY")
    if not api_key:
        raise RuntimeError("GLM_API_KEY 未在 .env 中设置，无法调用 glm-4.6v。")
    # 注意：zai-sdk 的 BaseClient 使用 api_key 作为参数名，而不是 apikey
    return ZhipuAiClient(api_key=api_key)




def build_system_prompt() -> str:
    return (
        "你是一名放射科主治医师，需要根据胸部 X 光片和放射学报告，"
        "输出结构化的中文解读结果，用于科普给普通患者。"
        "请严格按照 JSON 格式输出，键包括: "
        "`findings`(主要影像学发现), `impression`(综合印象), "
        "`recommendations`(随访或治疗建议), `chexpert_labels`(14 类标签判断)。"
        "其中 `chexpert_labels` 是一个对象，键为 14 个疾病名，值为 'positive'、'negative' 或 'uncertain'。"
        "请不要输出任何解释性文字或 Markdown，仅输出一个合法的 JSON 对象。"
    )




def build_user_prompt_for_text(report_text: str) -> str:
    return (
        "以下是胸部 X 光的英文放射学报告全文。"
        "请忽略可能出现的患者姓名等隐私信息，只关注影像学内容。\n\n"
        f"[放射学报告]\n{report_text}\n\n"
        "请根据以上报告内容完成结构化解读，并仅输出 JSON。"
    )




def build_user_prompt_for_multimodal(report_text: str) -> str:
    return (
        "我会提供一张胸部 X 光图像以及对应的英文放射学报告，"
        "请你结合图像和报告一起进行判断。如果图像与报告描述不符，"
        "以图像为主，但在印象中说明差异。\n\n"
        f"[放射学报告]\n{report_text}\n\n"
        "请严格按照前述 JSON 结构，仅输出 JSON。"
    )




def _extract_text_from_glm_content(content) -> str:
    """从 GLM 返回的 content 中提取纯文本。"""


    if isinstance(content, str):
        return content


    parts: List[str] = []
    try:
        for item in content:
            if isinstance(item, dict):
                t = item.get("text")
            else:
                t = getattr(item, "text", None)
            if isinstance(t, str):
                parts.append(t)
    except TypeError:
        pass


    if not parts:
        raise RuntimeError("GLM 返回结果中没有可解析的 text 内容。")
    return "".join(parts)




def _cleanup_json_text(text: str) -> str:
    """清理模型返回的文本，尽量提取出纯 JSON 字符串。"""


    text = text.strip()


    # 去掉 ```json ... ``` 或 ``` 包裹
    if text.startswith("```") and text.endswith("```"):

        inner = text.strip("`").strip()

        if inner.lower().startswith("json"):

            inner = inner[4:].lstrip()

        text = inner



    # 如果仍然不是以 { 开头，尝试从中间截取第一个 { 开始到最后一个 }
    if not text.lstrip().startswith("{"):

        first = text.find("{")

        last = text.rfind("}")

        if first != -1 and last != -1 and last > first:

            text = text[first : last + 1]



    return text.strip()





def _parse_json_from_glm_message(message) -> Dict:
    """从 ZhipuAiClient 的 message 对象中解析出 JSON。"""


    content = getattr(message, "content", None)
    if content is None:
        raise RuntimeError("GLM 返回结果中缺少 message.content。")


    raw_text = _extract_text_from_glm_content(content).strip()
    cleaned = _cleanup_json_text(raw_text)


    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        preview = raw_text[:200].replace("\n", " ")
        raise RuntimeError(f"GLM 返回内容不是合法 JSON，无法解析。前 200 字符为: {preview}") from e




def call_text_only_llm(report_text: str, config: ModelConfig) -> Dict:
    client = get_glm_client()
    sys_prompt = build_system_prompt()
    user_prompt = build_user_prompt_for_text(report_text)


    combined_text = sys_prompt + "\n\n" + user_prompt


    response = client.chat.completions.create(
        model=config.glm_model_text,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": combined_text,
                    }
                ],
            }
        ],
    )


    if not response.choices:
        raise RuntimeError("GLM 返回结果中缺少 choices。")
    return _parse_json_from_glm_message(response.choices[0].message)




def call_multimodal_llm(image_path: str, report_text: str, config: ModelConfig) -> Dict:
    """图像 + 报告多模态调用 glm-4.6v（使用 Base64 方式传图）。"""


    client = get_glm_client()
    sys_prompt = build_system_prompt()
    user_prompt = build_user_prompt_for_multimodal(report_text)


    # 读取本地图像并编码为 Base64 字符串（不带 data: 前缀）
    with open(image_path, "rb") as img_file:
        img_base = base64.b64encode(img_file.read()).decode("utf-8")


    combined_text = sys_prompt + "\n\n" + user_prompt


    response = client.chat.completions.create(
        model=config.glm_model_multimodal,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": img_base,
                        },
                    },
                    {
                        "type": "text",
                        "text": combined_text,
                    },
                ],
            }
        ],
    )


    if not response.choices:
        raise RuntimeError("GLM 返回结果中缺少 choices。")
    return _parse_json_from_glm_message(response.choices[0].message)




def compare_with_chexpert(pred_labels: Dict, gt_labels: Dict[str, Optional[float]]) -> Dict[str, Dict[str, Optional[str]]]:
    def map_gt(v: Optional[float]) -> Optional[str]:
        if v is None:
            return None
        if v > 0:
            return "positive"
        if v < 0:
            return "uncertain"
        return "negative"


    result: Dict[str, Dict[str, Optional[str]]] = {}
    for lab in CHEXPERT_LABELS:
        pred_val = None
        if isinstance(pred_labels, dict):
            raw = pred_labels.get(lab)
            if isinstance(raw, str):
                pred_val = raw.lower()
        gt_val = map_gt(gt_labels.get(lab)) if gt_labels else None
        result[lab] = {"pred": pred_val, "gt": gt_val}
    return result




def pretty_print_results(
    image_path: str,
    multimodal_res: Dict,
    text_res: Dict,
    chexpert_comparison_multi: Dict[str, Dict[str, Optional[str]]],
    chexpert_comparison_text: Dict[str, Dict[str, Optional[str]]],
) -> None:
    print("=" * 80)
    print(f"图像路径: {image_path}")
    print("=" * 80)


    print("[多模态模型（图像+报告）解读]\n")
    print("主要发现:", multimodal_res.get("findings"))
    print("综合印象:", multimodal_res.get("impression"))
    print("建议:", multimodal_res.get("recommendations"))
    print("\n与 CheXpert 标签对比 (multi):")
    for lab, vals in chexpert_comparison_multi.items():
        print(f"- {lab}: pred={vals['pred']}, gt={vals['gt']}")


    print("\n" + "-" * 80)
    print("[纯文本 LLM（仅报告）解读]\n")
    print("主要发现:", text_res.get("findings"))
    print("综合印象:", text_res.get("impression"))
    print("建议:", text_res.get("recommendations"))
    print("\n与 CheXpert 标签对比 (text-only):")
    for lab, vals in chexpert_comparison_text.items():
        print(f"- {lab}: pred={vals['pred']}, gt={vals['gt']}")



In [12]:
# 4. 把脚本的 main 逻辑改造成按步骤运行的单元格


# 在这里配置一张测试图像和对应的报告（可以手动填写或从文件读取）


config = ModelConfig()



# 示例：从指定目录中自动选择一张图像作为示例


example_dir = DATA_ROOT / "files" / "p10" / "p10000032" / "s56699142"


example_image_path = None
for pattern in ("*.jpg", "*.jpeg", "*.png"):
    candidates = list(example_dir.glob(pattern))
    if candidates:
        example_image_path = candidates[0]
        break


if example_image_path is None:
    raise FileNotFoundError(f"在目录 {example_dir} 下没有找到示例图像文件，请检查数据集路径。")


print("示例图像路径:", example_image_path)


# 你可以直接在这里写入一小段英文报告，或改为从 txt 读取
example_report_text = """PA and lateral chest radiographs demonstrate hyperinflated lungs with flattening of the diaphragms. No focal consolidation or pleural effusion is identified."""


# 解析 CheXpert 标签
ids = parse_ids_from_path(str(example_image_path))
chexpert_labels: Dict[str, Optional[float]] = {}
if ids is not None:
    df_chexpert = load_chexpert_labels(config)
    chexpert_labels = get_labels_for_study(df_chexpert, ids["subject_id"], ids["study_id"])
else:
    print("警告：未能从路径解析出 subject_id / study_id，将无法对比 CheXpert 标签。")


print("CheXpert 标签(原始数值):")
print({k: v for k, v in chexpert_labels.items() if v is not None})



示例图像路径: d:\Documents\SRTP_main\SRTP_Data\files\p10\p10000032\s56699142\ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c.jpg
CheXpert 标签(原始数值):
{'No Finding': 1.0}


In [13]:
# 5. 在 Notebook 中进行一次多模态 vs 纯文本解读对比


# 注意：运行前需要在项目根目录的 .env 中配置 GLM_API_KEY（zai-sdk 会使用默认 API 地址）


# 简单验证图像能否打开
_ = Image.open(example_image_path).convert("RGB")


multimodal_res = call_multimodal_llm(str(example_image_path), example_report_text, config)
text_res = call_text_only_llm(example_report_text, config)


multi_cmp = compare_with_chexpert(multimodal_res.get("chexpert_labels", {}), chexpert_labels)
text_cmp = compare_with_chexpert(text_res.get("chexpert_labels", {}), chexpert_labels)


pretty_print_results(str(example_image_path), multimodal_res, text_res, multi_cmp, text_cmp)


图像路径: d:\Documents\SRTP_main\SRTP_Data\files\p10\p10000032\s56699142\ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c.jpg
[多模态模型（图像+报告）解读]

主要发现: 胸部X光片显示双肺过度充气，膈肌变平；未见局灶性实变或胸腔积液。
综合印象: 双肺过度充气，膈肌变平，提示可能存在肺过度充气状态（如慢性阻塞性肺疾病等），未见明显感染或胸腔积液。
建议: 建议结合临床症状及肺功能检查进一步评估；避免吸烟，遵医嘱进行相应治疗，定期随访。

与 CheXpert 标签对比 (multi):
- Atelectasis: pred=negative, gt=None
- Cardiomegaly: pred=negative, gt=None
- Consolidation: pred=negative, gt=None
- Edema: pred=negative, gt=None
- Enlarged Cardiomediastinum: pred=None, gt=None
- Fracture: pred=None, gt=None
- Lung Lesion: pred=None, gt=None
- Lung Opacity: pred=None, gt=None
- Pleural Effusion: pred=negative, gt=None
- Pneumonia: pred=negative, gt=None
- Pneumothorax: pred=negative, gt=None
- Pleural Other: pred=None, gt=None
- Support Devices: pred=None, gt=None
- No Finding: pred=None, gt=positive

--------------------------------------------------------------------------------
[纯文本 LLM（仅报告）解读]

主要发现: 胸部 X 光片显示肺过度充气，膈肌变平。未见局灶性实变或胸腔积液。
综合印象: 符合肺气肿表现，未见急性感染或胸腔积液。
建议:

In [ ]:
# 6. 简单测试与断言（可根据需要扩展）

# 示例：检查标签对比函数的输出结构
sample_pred = {lab: "positive" for lab in CHEXPERT_LABELS}
sample_gt = {lab: 1.0 for lab in CHEXPERT_LABELS}

cmp_example = compare_with_chexpert(sample_pred, sample_gt)

assert set(cmp_example.keys()) == set(CHEXPERT_LABELS)
assert all(v["pred"] == "positive" and v["gt"] == "positive" for v in cmp_example.values())

print("简单测试通过，compare_with_chexpert 输出结构正常。")

In [ ]:
# 7. （可选）将 Notebook 中的关键函数同步回 .py 文件

from textwrap import dedent

module_path = PROJECT_ROOT / "code" / "multimodal_cxr_report_from_notebook.py"

module_code = dedent('''
from dataclasses import dataclass
from typing import Dict, List, Optional
import json
import os
import re

import pandas as pd
from PIL import Image
from dotenv import load_dotenv

try:
    from openai import OpenAI  # type: ignore
except Exception:
    OpenAI = None  # type: ignore

CHEXPERT_LABELS: List[str] = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "Pleural Other",
    "Support Devices",
    "No Finding",
]


@dataclass
class ModelConfig:
    openai_model_multimodal: str = "gpt-4.1-mini"
    openai_model_text: str = "gpt-4.1-mini"
    data_root: str = "{data_root}"
    chexpert_csv: str = "{chexpert_csv}"


def parse_ids_from_path(image_path: str) -> Optional[Dict[str, str]]:
    basename = os.path.basename(image_path)
    m = re.match(r"(\\d+)_(\\d+)_", basename)
    if not m:
        return None
    subject_id, study_id = m.group(1), m.group(2)
    return {{"subject_id": subject_id, "study_id": study_id}}


# 其余函数可按需从 Notebook 拷贝补充...
''').format(data_root=str(DATA_ROOT), chexpert_csv=str(DATA_ROOT / "mimic-cxr-2.0.0-chexpert.csv"))

with open(module_path, "w", encoding="utf-8") as f:
    f.write(module_code)

print("已将基础骨架写入:", module_path)
print("如需完整同步，可以继续从 Notebook 复制更多函数定义。")